### Build a Simple LLM Application with LCEL
In this quickstart we'll show you how to build a simple LLM application with LangChain. This application will translate text from English into another language. This is a relatively simple LLM application - it's just a single LLM call plus some prompting. Still, this is a great way to get started with LangChain - a lot of features can be built with just some prompting and an LLM call!

After seeing this video, you'll have a high level overview of:

- Using language models

- Using PromptTemplates and OutputParsers

- Using LangChain Expression Language (LCEL) to chain components together

- Debugging and tracing your application using LangSmith

- Deploying your application with LangServe

In [ ]:
### Open AI API Key and Open Source models--Llama3,Gemma2,mistral--Groq
from dotenv import load_dotenv
import os

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
groq_api_key

'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'

In [ ]:
from langchain_groq import ChatGroq
model = ChatGroq(model = "llama-3.1-8b-instant", groq_api_key = groq_api_key)
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000023837CE5F60>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002385EA92200>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [4]:
import requests
import os

url = "https://api.groq.com/openai/v1/models"

headers = {
    "Authorization": f"Bearer {groq_api_key}",
    "Content-Type": "application/json"
}

response = requests.get(url, headers=headers)

print(response.json())

{'object': 'list', 'data': [{'id': 'canopylabs/orpheus-arabic-saudi', 'object': 'model', 'created': 1765926439, 'owned_by': 'Canopy Labs', 'active': True, 'context_window': 4000, 'public_apps': None, 'max_completion_tokens': 50000}, {'id': 'meta-llama/llama-prompt-guard-2-22m', 'object': 'model', 'created': 1748632101, 'owned_by': 'Meta', 'active': True, 'context_window': 512, 'public_apps': None, 'max_completion_tokens': 512}, {'id': 'allam-2-7b', 'object': 'model', 'created': 1737672203, 'owned_by': 'SDAIA', 'active': True, 'context_window': 4096, 'public_apps': None, 'max_completion_tokens': 4096}, {'id': 'openai/gpt-oss-120b', 'object': 'model', 'created': 1754408224, 'owned_by': 'OpenAI', 'active': True, 'context_window': 131072, 'public_apps': None, 'max_completion_tokens': 65536}, {'id': 'meta-llama/llama-4-scout-17b-16e-instruct', 'object': 'model', 'created': 1743874824, 'owned_by': 'Meta', 'active': True, 'context_window': 131072, 'public_apps': None, 'max_completion_tokens':

In [10]:
from langchain_core.messages import HumanMessage,SystemMessage

messages=[
    SystemMessage(content = "Translate the following from English to French"),
    HumanMessage(content = "Hello How are you?")
]

result = model.invoke(messages)

In [11]:
result

AIMessage(content='Bonjour Comment allez-vous ?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 47, 'total_tokens': 54, 'completion_time': 0.012650593, 'completion_tokens_details': None, 'prompt_time': 0.002238674, 'prompt_tokens_details': None, 'queue_time': 0.018237814, 'total_time': 0.014889267}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_03e8423237', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d6716-c9b8-7702-9ab5-bd09a1a0e4a0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 47, 'output_tokens': 7, 'total_tokens': 54})

In [12]:
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()
parser.invoke(result)

'Bonjour Comment allez-vous ?'

In [13]:
### Using LCEL- chain the components
chain = model | parser
chain.invoke(messages)

'Bonjour Comment ça va ?'

In [ ]:
### Prompt Templates
from langchain_core.prompts import ChatPromptTemplate
    
generic_template="Translate the following into {language}:"

prompt = ChatPromptTemplate.from_messages(
    [("system", generic_template),("user","{text}")]
)

In [15]:
result = prompt.invoke({"language":"French","text":"Hello"})
result.to_messages()

[SystemMessage(content='Translate the following into French:', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello', additional_kwargs={}, response_metadata={})]

In [ ]:
## Chaining together components with LCEL
chain = prompt | model | parser
chain.invoke({"language":"French","text":"Hello"})

'Bonjour \n'